# 01 — Dataset Understanding

**Project:** Customer Segmentation Using Unsupervised Machine Learning  
**Dataset:** UCI Online Retail  
**Unit of analysis at source level:** Transaction line  
**Notebook stage:** Phase 2 — Dataset Understanding and Exploratory Data Analysis

## Purpose

This notebook establishes a baseline understanding of the untouched Online Retail dataset before any cleaning, filtering or feature engineering is performed.

## Objectives

1. Load the original workbook without modifying its records.
2. inspect its dimensions and original variables.
3. Review the data types and date coverage.
4. Establish the initial facts needed to guide the subsequent data-quality assessment.

**Scope note:** This notebook is descriptive only. No records are removed and no analytical customer-level features are created here. The raw dataset is preserved for all subsequent stages.

## 1.0 Load the original dataset

Load the workbook through the project's shared `load_online_retail` function. The source file is read as-is; this notebook does not alter or overwrite it.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loading import load_online_retail

df = load_online_retail()

print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
display(df.head())

Shape: 541,909 rows × 8 columns


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


### Interpretation: Initial load

The original workbook has loaded successfully with **541,909 rows and eight columns**. The displayed sample provides an initial view of the transaction-level structure. It is used only for orientation; decisions about exclusions or data cleaning will be based on systematic checks in the data-quality assessment, not on the first few records.

## 1.1 Original variable dictionary

The table documents the eight variables in the original transaction-level dataset and explains their intended analytical roles.

Two distinctions are important for the project design:

- **CustomerID** is an identifier used to aggregate transactions to the customer level; it is not a numerical clustering feature.
- **Country** is retained for geographic profiling after clustering rather than included as a primary clustering feature.

The expected analytical types describe how each variable will be treated in the analysis; they do not necessarily match the storage dtype in the raw workbook.

In [2]:
variable_dictionary = pd.DataFrame(
    [
        [
            "InvoiceNo",
            "Categorical / string",
            "Transaction identifier",
            "Count distinct invoices; inspect C-prefixed cancellation records",
        ],
        [
            "StockCode",
            "Categorical / string",
            "Product identifier",
            "Count distinct products for product diversity",
        ],
        [
            "Description",
            "Text",
            "Product information",
            "Optional product-level interpretation",
        ],
        [
            "Quantity",
            "Integer",
            "Transaction feature",
            "Purchase volume and negative-quantity records",
        ],
        [
            "InvoiceDate",
            "Datetime",
            "Transaction feature",
            "Monthly trends, recency and purchase intervals",
        ],
        [
            "UnitPrice",
            "Continuous numeric",
            "Transaction feature",
            "Calculate line value with Quantity × UnitPrice",
        ],
        [
            "CustomerID",
            "Identifier (stored numerically)",
            "Customer identifier",
            "Customer-level aggregation key only",
        ],
        [
            "Country",
            "Categorical / string",
            "Profiling variable",
            "Geographic comparisons after clustering",
        ],
    ],
    columns=[
        "Variable",
        "Expected analytical type",
        "Role",
        "Analytical purpose",
    ],
)

display(variable_dictionary)

variable_dictionary.to_csv(
    PROJECT_ROOT / "reports" / "phase_2" / "tables" / "variable_dictionary.csv",
    index=False,
)

,Variable,Expected analytical type,Role,Analytical purpose
0,InvoiceNo,Categorical / string,Transaction identifier,Count distinct invoices; inspect C-prefixed ca...
1,StockCode,Categorical / string,Product identifier,Count distinct products for product diversity
2,Description,Text,Product information,Optional product-level interpretation
3,Quantity,Integer,Transaction feature,Purchase volume and negative-quantity records
4,InvoiceDate,Datetime,Transaction feature,"Monthly trends, recency and purchase intervals"
5,UnitPrice,Continuous numeric,Transaction feature,Calculate line value with Quantity × UnitPrice
6,CustomerID,Identifier (stored numerically),Customer identifier,Customer-level aggregation key only
7,Country,Categorical / string,Profiling variable,Geographic comparisons after clustering


### Interpretation: Variable roles

The variable dictionary distinguishes transaction identifiers, product identifiers, descriptive text, purchase measures, customer identifiers and geographic information. This distinction is important because identifiers should not be treated as continuous numerical measurements in clustering.

The dictionary is also saved as `reports/phase_2/tables/variable_dictionary.csv` so that the variable definitions can be reused in the data and preprocessing documentation.

## 1.2 Data types, dataset dimensions and date coverage

This section checks how pandas has interpreted the original columns and establishes the dataset's basic scale and observation period.

These checks help identify variables that may require type conversion or special handling during the data-quality assessment.

In [3]:
display(df.dtypes.rename("pandas_dtype").to_frame())

print(
    "Date range:",
    df["InvoiceDate"].min(),
    "to",
    df["InvoiceDate"].max(),
)
print(
    "Unique customers (non-missing):",
    f"{df['CustomerID'].nunique():,}",
)
print("Unique invoices:", f"{df['InvoiceNo'].nunique():,}")
print("Unique StockCodes:", f"{df['StockCode'].nunique():,}")
print("Countries:", f"{df['Country'].nunique():,}")

,pandas_dtype
InvoiceNo,object
StockCode,object
Description,object
Quantity,int64
InvoiceDate,datetime64[us]
UnitPrice,float64
CustomerID,float64
Country,str


Date range: 2010-12-01 08:26:00 to 2011-12-09 12:50:00
Unique customers (non-missing): 4,372
Unique invoices: 25,900
Unique StockCodes: 4,070
Countries: 38


### Interpretation: Dataset structure and coverage

The initial inspection confirms that the workbook contains **541,909 transaction-level records across eight variables**. The observation period runs from **1 December 2010 to 9 December 2011**, providing approximately one year of transaction history.

The dataset contains **4,372 distinct non-missing customer identifiers**, **25,900 distinct invoice numbers**, **4,070 distinct stock codes** and **38 countries**. These are raw-dataset counts; they should not be confused with the final customer population used in clustering, which will be established after data-quality checks and preprocessing.

The storage types are broadly consistent with the variables' roles: `InvoiceDate` is represented as a datetime, `Quantity` as an integer and `UnitPrice` as a floating-point number. `CustomerID` is stored numerically because missing values are present, but it will be treated as an identifier rather than a continuous measure.

These findings establish the dataset's structure. They do not yet establish whether all rows represent valid completed purchases. Missing identifiers, cancellations, returns, duplicates and unusual prices or quantities require separate investigation in the next notebook.

## 1.3 Notebook summary and transition

### Key findings

- The untouched dataset contains **541,909 rows and eight original variables**.
- The records span **1 December 2010 to 9 December 2011**.
- There are **4,372 distinct non-missing CustomerIDs**, **25,900 invoice numbers**, **4,070 StockCodes** and **38 countries**.
- `CustomerID` will be used only as the customer-level aggregation key.
- `Country` will be used for post-clustering profiling, not as a primary clustering input.
- The raw dataset has been inspected but has not been cleaned or filtered in this notebook.

### Analytical implication

The data contains the transaction, product, quantity, price and date information needed to construct customer-level purchasing behaviour features. However, the raw counts cannot yet be treated as the final modelling sample. The validity and treatment of transaction records must first be assessed.

### Next step

Notebook 02, **Data Quality Assessment**, will examine missing values, duplicates, cancellations, negative quantities, zero or negative prices and other potential data-quality issues. The findings will inform documented preprocessing decisions in the subsequent stage.